# Polígonos

Descarga polígonos para cada manzano a nivel nacional del geoportal del INE.

In [1]:
import requests
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
from io import BytesIO
from openpyxl import load_workbook
import json
from IPython.display import clear_output
from pathlib import Path

( Una lista de municipios tomada de microdatos del censo 2024 )

In [2]:
municipios = pd.read_csv("recursos/municipios.csv", index_col=["codigo"])
municipios

,municipio,departamento,poblacion
codigo,,,
10101,Sucre,Chuquisaca,296746
10102,Yotala,Chuquisaca,9256
10103,Poroma,Chuquisaca,13649
10201,Azurduy,Chuquisaca,7903
10202,Tarvita,Chuquisaca,16812
...,...,...,...
90401,Santa Rosa,Pando,2889
90402,Ingavi,Pando,2600
90501,Nueva Esperanza,Pando,1598


Cabezales para cada consulta.

In [3]:
headers = {
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:144.0) Gecko/20100101 Firefox/144.0',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.5',
    'Content-Type': 'application/json',
    'Origin': 'https://geoportal.ine.gob.bo',
    'Sec-GPC': '1',
    'Connection': 'keep-alive',
    'Referer': 'https://geoportal.ine.gob.bo/',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-site',
}

In [4]:
def refresh_headers():
    response = requests.post(
        "https://wgeoportal.ine.gob.bo/api/v1/geoportal/registroSesion",
    )
    r = response.json()
    token = r["session_token"]
    headers["Authorization"] = f"Bearer {token}"
    headers["X-Session-Token"] = token

Descargar polígonos para un municipio.

In [29]:
def get_comunidades_municipio(codigo):
    response = requests.post(
        "https://wgeoportal.ine.gob.bo/api/v1/selecccion/depMunSeleccionadoPunto",
        headers=headers,
        json={
            "id": "0" + str(codigo),
            "tipo": "municipio",
        },
    )

    while True:
        if response.status_code == 200:
            municipio = municipios.loc[codigo][["departamento", "municipio"]].to_dict()

            if response.json():
                features = [
                    {
                        "type": "Feature",
                        "properties": {
                            **municipio,
                            **{key: feature[key] for key in ["id", "nombre"]},
                        },
                        "geometry": eval(feature["geojson"]),
                    }
                    for feature in response.json()["datos"]
                ]
            else:
                features = []

            return features
        else:
            refresh_headers()

Descargar polígonos para todos los municipios.

In [32]:
def get_comunidades():
    features = []
    for codigo in tqdm(municipios.index):
        features.extend(get_comunidades_municipio(codigo))
    gdf = gpd.GeoDataFrame.from_features({
        "type": "FeatureCollection",
        "features": features
    })
    return gdf[["departamento", "municipio", "nombre", "id", "geometry"]]

Inicializar cabezales y descargar!

In [33]:
refresh_headers()

In [34]:
gdf = get_comunidades()

100%|██████████| 342/342 [01:19<00:00,  4.32it/s]


In [35]:
gdf

,departamento,municipio,nombre,id,geometry
0,Chuquisaca,Sucre,KARA HUASI,04099136304-D,POINT (-65.19151 -19.18249)
1,Chuquisaca,Sucre,COMUNIDAD C.O. TALULA,04100982768-D,POINT (-65.45218 -19.12024)
2,Chuquisaca,Sucre,COMUNIDAD C.O. LECOPAYA,04106914964-D,POINT (-65.37253 -19.12821)
3,Chuquisaca,Sucre,COMUNIDAD C.O. HUMACA,04111042175-D,POINT (-65.46666 -19.10093)
4,Chuquisaca,Sucre,COMUNIDAD C.O. PURUNQUILA,04111281903-D,POINT (-65.41133 -19.11257)
...,...,...,...,...,...
21143,Pando,Santos Mercado,COMUNIDAD CAMPESINA CHIWAWA,12032530522-D,POINT (-66.48674 -10.20737)
21144,Pando,Santos Mercado,COMUNIDAD PUERTO BOLIVAR,12136933247-D,POINT (-66.50058 -10.11354)
21145,Pando,Santos Mercado,TAJIBO,12243450456-D,POINT (-66.47078 -10.00303)
21146,Pando,Santos Mercado,7 DE ENERO,12325789475-D,POINT (-66.21111 -9.83757)


Guardar como geoparquet.

In [36]:
gdf.to_parquet("datos/comunidades.parquet")